In [2]:
import numpy as np
from scipy.optimize import least_squares


# UR5 DH PARAMETERS (meters)


d1 = 0.089159
a2 = -0.42500
a3 = -0.39225
d4 = 0.10915
d5 = 0.09465
d6 = 0.08230

DH = [
    [0, np.pi/2, d1],
    [a2, 0, 0],
    [a3, 0, 0],
    [0, np.pi/2, d4],
    [0, -np.pi/2, d5],
    [0, 0, d6]
]


# DH TRANSFORMATION MATRIX


def dh_transform(a, alpha, d, theta):

    return np.array([
        [np.cos(theta),
         -np.sin(theta)*np.cos(alpha),
         np.sin(theta)*np.sin(alpha),
         a*np.cos(theta)],

        [np.sin(theta),
         np.cos(theta)*np.cos(alpha),
         -np.cos(theta)*np.sin(alpha),
         a*np.sin(theta)],

        [0,
         np.sin(alpha),
         np.cos(alpha),
         d],

        [0,0,0,1]
    ])



# FORWARD KINEMATICS


def forward_kinematics(q):

    T = np.eye(4)

    for i in range(6):

        a, alpha, d = DH[i]

        T = T @ dh_transform(
            a,
            alpha,
            d,
            q[i]
        )

    return T



# ROTATION MATRIX TO ROTATION VECTOR


def rotation_error(R_des, R_cur):

    R_err = R_des @ R_cur.T

    angle = np.arccos(
        np.clip((np.trace(R_err)-1)/2,
        -1.0,
        1.0)
    )

    if abs(angle) < 1e-8:
        return np.zeros(3)

    axis = np.array([
        R_err[2,1]-R_err[1,2],
        R_err[0,2]-R_err[2,0],
        R_err[1,0]-R_err[0,1]
    ])/(2*np.sin(angle))

    return angle*axis



# INVERSE KINEMATICS


def inverse_kinematics(T_desired):

    def error_function(q):

        T = forward_kinematics(q)

        pos_error = (
            T_desired[:3,3] -
            T[:3,3]
        )

        ori_error = rotation_error(
            T_desired[:3,:3],
            T[:3,:3]
        )

        return np.concatenate(
            [pos_error, ori_error]
        )

    q0 = np.zeros(6)

    solution = least_squares(
        error_function,
        q0,
        method='lm'
    )

    return solution.x



# NUMERICAL JACOBIAN


def compute_jacobian(q):

    J = np.zeros((6,6))

    delta = 1e-6

    T0 = forward_kinematics(q)

    p0 = T0[:3,3]

    for i in range(6):

        q_new = q.copy()

        q_new[i] += delta

        T1 = forward_kinematics(q_new)

        p1 = T1[:3,3]

        dp = (p1-p0)/delta

        dR = rotation_error(
            T1[:3,:3],
            T0[:3,:3]
        )/delta

        J[:,i] = np.hstack((dp,dR))

    return J



# SINGULARITY CHECK


def check_singularity(J):

    rank = np.linalg.matrix_rank(J)

    if rank < 6:
        return True

    return False


# IK VERIFICATION


def verify_ik(T_desired, q_ik):

    T_fk = forward_kinematics(q_ik)

    pos_error = np.linalg.norm(
        T_desired[:3,3] -
        T_fk[:3,3]
    )

    ori_error = np.linalg.norm(
        rotation_error(
            T_desired[:3,:3],
            T_fk[:3,:3]
        )
    )

    return pos_error, ori_error



# MAIN PROGRAM

if __name__ == "__main__":

    print("\nUR5 IK VERIFICATION TOOL\n")

    q_test = np.radians(
        [30,-45,60,90,-30,45]
    )

    print("Input Joint Angles (deg):")

    print(np.degrees(q_test))

    T_desired = forward_kinematics(q_test)

    print("\nDesired Pose:")

    print(np.round(T_desired,4))

    q_ik = inverse_kinematics(T_desired)

    print("\nIK Solution (deg):")

    print(np.round(
        np.degrees(q_ik),
        3
    ))

    pos_err, ori_err = verify_ik(
        T_desired,
        q_ik
    )

    print("\nVerification Results")

    print(
        "Position Error:",
        pos_err,
        "m"
    )

    print(
        "Orientation Error:",
        ori_err,
        "rad"
    )

    J = compute_jacobian(q_ik)

    print("\nJacobian Matrix:\n")

    print(np.round(J,4))

    singular = check_singularity(J)

    print("\nSingularity Status:")

    if singular:
        print("SINGULAR CONFIGURATION")
    else:
        print("NON-SINGULAR CONFIGURATION")

    print("\nIK VALID")



UR5 IK VERIFICATION TOOL

Input Joint Angles (deg):
[ 30. -45.  60.  90. -30.  45.]

Desired Pose:
[[-0.9055 -0.2775  0.3209 -0.4282]
 [-0.1146 -0.5684 -0.8147 -0.4556]
 [ 0.4085 -0.7745  0.483   0.3524]
 [ 0.      0.      0.      1.    ]]

IK Solution (deg):
[   30.    -2147.651  4260.    -2007.349   -30.       45.   ]

Verification Results
Position Error: 5.500368015904955e-15 m
Orientation Error: 0.0 rad

Jacobian Matrix:

[[ 0.4556 -0.228  -0.3067 -0.0556  0.0366  0.    ]
 [-0.4282 -0.1316 -0.1771 -0.0321 -0.0264  0.    ]
 [ 0.     -0.5986 -0.1835  0.0808 -0.0688  0.    ]
 [ 0.      0.5     0.5     0.5     0.8365  0.3209]
 [ 0.     -0.866  -0.866  -0.866   0.483  -0.8147]
 [ 1.     -0.     -0.     -0.      0.2588  0.483 ]]

Singularity Status:
NON-SINGULAR CONFIGURATION

IK VALID
